# Retail Sales Data Analysis and Revenue Forecasting Using AI
**Prepared by:** Abhishek  
**Program:** IBM SkillsBuild Data Analytics with AI Academic Internship  
**Organization:** BharatCares in association with AICTE

## Project overview
This notebook creates a reproducible synthetic retail-sales dataset, explores trends and product performance, and trains a machine-learning model to estimate daily revenue. The generated data is illustrative and is not a real company's confidential sales data.

### Objectives
- Clean and validate the data.
- Summarize revenue, profit, units, product and regional performance.
- Visualize trends and seasonality.
- Build and evaluate a Random Forest regression model.
- Explain limitations and practical next steps.


## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)


## 2. Generate or load the dataset
The CSV file supplied with this project is generated with a fixed random seed. If it is beside this notebook, load it; otherwise the following cell regenerates the same type of data.

In [ ]:
from pathlib import Path

DATA_PATH = Path("retail_sales_dataset.csv")
if DATA_PATH.exists():
    sales = pd.read_csv(DATA_PATH, parse_dates=["date"])
else:
    rng = np.random.default_rng(RANDOM_STATE)
    dates = pd.date_range("2023-01-01", "2025-12-31", freq="D")
    products = ["Laptop", "Smartphone", "Headphones", "Smartwatch", "Tablet"]
    categories = {"Laptop":"Computers", "Smartphone":"Mobiles", "Headphones":"Accessories",
                  "Smartwatch":"Wearables", "Tablet":"Computers"}
    base_price = {"Laptop":65000, "Smartphone":28000, "Headphones":2500,
                  "Smartwatch":8000, "Tablet":22000}
    base_units = {"Laptop":5, "Smartphone":12, "Headphones":30, "Smartwatch":10, "Tablet":7}
    regions = ["North", "South", "East", "West", "Central"]
    rows = []
    for d in dates:
        seasonal = 1 + 0.18*np.sin(2*np.pi*(d.dayofyear/365.25)) + (0.35 if d.month in [10,11,12] else 0)
        weekend = 1.15 if d.dayofweek >= 5 else 1.0
        for p in products:
            units = max(0, int(rng.poisson(base_units[p]*seasonal*weekend)))
            price = base_price[p]*(1+rng.normal(0,0.025))
            discount = float(rng.choice([0,.05,.10,.15,.20], p=[.35,.2,.25,.15,.05]))
            revenue = units*price*(1-discount)
            cost_rate = {"Laptop":.78,"Smartphone":.72,"Headphones":.48,"Smartwatch":.60,"Tablet":.70}[p]
            cost = units*price*cost_rate
            rows.append([d,p,categories[p],str(rng.choice(regions)),units,round(price,2),discount,
                         round(revenue,2),round(cost,2),round(revenue-cost,2)])
    sales = pd.DataFrame(rows, columns=["date","product","category","region","units_sold",
        "unit_price_inr","discount_rate","revenue_inr","cost_inr","profit_inr"])
    sales.to_csv(DATA_PATH, index=False)

sales.head()


## 3. Data quality checks

In [ ]:
print("Shape:", sales.shape)
print("\\nData types:")
print(sales.dtypes)
print("\\nMissing values:")
print(sales.isna().sum())
print("\\nDuplicate rows:", sales.duplicated().sum())
print("\\nDate range:", sales["date"].min(), "to", sales["date"].max())
assert sales["date"].notna().all()
assert (sales["units_sold"] >= 0).all()
assert (sales["revenue_inr"] >= 0).all()


## 4. Descriptive statistics and KPIs

In [ ]:
kpis = {
    "Total revenue (INR)": sales["revenue_inr"].sum(),
    "Total profit (INR)": sales["profit_inr"].sum(),
    "Units sold": sales["units_sold"].sum(),
    "Average order-line revenue (INR)": sales["revenue_inr"].mean(),
    "Profit margin (%)": 100*sales["profit_inr"].sum()/sales["revenue_inr"].sum()
}
pd.Series(kpis).round(2)


## 5. Product and regional analysis

In [ ]:
product_summary = sales.groupby(["product","category"], as_index=False).agg(
    revenue_inr=("revenue_inr","sum"),
    profit_inr=("profit_inr","sum"),
    units_sold=("units_sold","sum")
).sort_values("revenue_inr", ascending=False)
display(product_summary)

region_summary = sales.groupby("region", as_index=False).agg(
    revenue_inr=("revenue_inr","sum"),
    profit_inr=("profit_inr","sum"),
    units_sold=("units_sold","sum")
).sort_values("revenue_inr", ascending=False)
display(region_summary)


In [ ]:
product_summary.set_index("product")["revenue_inr"].sort_values().plot(kind="barh", figsize=(8,4))
plt.title("Total Revenue by Product")
plt.xlabel("Revenue (INR)")
plt.tight_layout()
plt.show()

region_summary.set_index("region")["revenue_inr"].sort_values().plot(kind="bar", figsize=(7,4))
plt.title("Total Revenue by Region")
plt.ylabel("Revenue (INR)")
plt.tight_layout()
plt.show()


## 6. Time-series analysis

In [ ]:
daily = sales.groupby("date", as_index=False).agg(
    revenue_inr=("revenue_inr","sum"),
    profit_inr=("profit_inr","sum"),
    units_sold=("units_sold","sum")
)
daily["month"] = daily["date"].dt.to_period("M").dt.to_timestamp()
monthly = daily.groupby("month", as_index=False).agg(
    revenue_inr=("revenue_inr","sum"),
    profit_inr=("profit_inr","sum"),
    units_sold=("units_sold","sum")
)
monthly.tail()


In [ ]:
plt.figure(figsize=(11,4))
plt.plot(monthly["month"], monthly["revenue_inr"], linewidth=1.8)
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue (INR)")
plt.grid(alpha=.25)
plt.tight_layout()
plt.show()


## 7. AI / machine-learning model
A Random Forest regressor estimates daily revenue using calendar features and the previous day's, previous week's, and rolling average revenue. The time-ordered split avoids training on future dates and testing on earlier dates.

In [ ]:
daily = daily.sort_values("date").copy()
daily["day_of_week"] = daily["date"].dt.dayofweek
daily["month_num"] = daily["date"].dt.month
daily["day_of_year"] = daily["date"].dt.dayofyear
daily["is_weekend"] = (daily["day_of_week"] >= 5).astype(int)
daily["lag_1"] = daily["revenue_inr"].shift(1)
daily["lag_7"] = daily["revenue_inr"].shift(7)
daily["rolling_mean_7"] = daily["revenue_inr"].shift(1).rolling(7).mean()
model_data = daily.dropna().copy()
features = ["day_of_week","month_num","day_of_year","is_weekend","lag_1","lag_7","rolling_mean_7"]
X = model_data[features]
y = model_data["revenue_inr"]
split_idx = int(len(model_data)*0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
model = RandomForestRegressor(n_estimators=250, max_depth=12, min_samples_leaf=2,
                              random_state=RANDOM_STATE, n_jobs=-1)
model.fit(X_train, y_train)
pred = model.predict(X_test)
metrics = {
    "MAE (INR)": mean_absolute_error(y_test, pred),
    "RMSE (INR)": mean_squared_error(y_test, pred)**0.5,
    "R-squared": r2_score(y_test, pred)
}
pd.Series(metrics).round(3)


In [ ]:
plt.figure(figsize=(11,4))
plt.plot(model_data["date"].iloc[split_idx:], y_test.values, label="Actual", linewidth=1.5)
plt.plot(model_data["date"].iloc[split_idx:], pred, label="Predicted", linewidth=1.3)
plt.title("Actual vs Predicted Daily Revenue (Test Period)")
plt.xlabel("Date")
plt.ylabel("Revenue (INR)")
plt.legend()
plt.grid(alpha=.25)
plt.tight_layout()
plt.show()

pd.Series(model.feature_importances_, index=features).sort_values().plot(kind="barh", figsize=(7,4))
plt.title("Model Feature Importance")
plt.tight_layout()
plt.show()


## 8. Findings and interpretation
Use the calculated tables and plots above to identify the strongest product and region, seasonal peaks, and the forecasting error. The dataset is synthetic, so these findings demonstrate an analytical workflow rather than making claims about a real retailer.

### How to interpret the model
- **MAE** is the average absolute prediction error in INR; lower is better.
- **RMSE** penalizes large errors more strongly than MAE.
- **R-squared** measures the proportion of variation explained by the model; it can be negative on poor forecasts.
- Feature importance describes the model's reliance on inputs, not proof of causation.

## 9. Limitations and ethical considerations
1. Data is simulated, with assumptions about product prices, unit volumes, discounts, costs, and seasonality.
2. No marketing spend, inventory, holidays, competitor prices, or customer-level behavior is included.
3. Model performance on synthetic data does not establish performance on a live business dataset.
4. A real deployment should validate data quality, monitor drift, compare with a seasonal-naive baseline, and avoid using personal data without a lawful purpose and appropriate safeguards.
5. Forecasts should support, not replace, human decisions.

## 10. Conclusion and future scope
The project demonstrates a full data analytics workflow: dataset creation, quality checks, KPI calculation, product/region analysis, visual storytelling, and a time-ordered machine-learning evaluation. Future work could use an approved real dataset, add holidays and promotions, compare gradient boosting models, and produce a dashboard in Power BI or Tableau.

## Reproducibility
Install packages from `requirements.txt`, place `retail_sales_dataset.csv` beside this notebook, and run all cells from top to bottom. The random seed is fixed at 42.
